<style>
/* 교재형 노트북: 기본값보다 조금 작고 촘촘하게 표시합니다. */
.jp-MarkdownOutput, .markdown-body {
    font-size: 0.94em !important;
    line-height: 1.68 !important;
}
.jp-MarkdownOutput h1, .markdown-body h1 { font-size: 1.72em !important; }
.jp-MarkdownOutput h2, .markdown-body h2 { font-size: 1.38em !important; }
.jp-MarkdownOutput h3, .markdown-body h3 { font-size: 1.14em !important; }
.jp-CodeCell, .jp-OutputArea-output, .cell.code_cell, .output_area {
    font-size: 0.92em !important;
}
.jp-MarkdownOutput table, .markdown-body table { font-size: 0.92em !important; }
.jp-MarkdownOutput blockquote, .markdown-body blockquote {
    border-left: 4px solid #4c78a8;
    padding-left: 0.9em;
    color: #4b5563;
}
</style>

# AlexNet 안으로 들어가기: 합성곱, 특징맵, 풀링, 수용영역

이 노트북은 퍼셉트론과 MLP 다음에 CNN이 왜 필요했는지 AlexNet의 내부 출력을 직접 보며 이해하는 실습입니다. 완성된 정확도만 보는 대신, 한 장의 이미지가 층을 지날 때 **모양과 정보의 표현 방식이 어떻게 변하는지** 추적합니다.

학습 흐름은 다음과 같습니다.

`원본 이미지 → 합성곱 → 활성화 → 특징맵 → Pooling → 넓어진 수용영역 → 분류 점수`

> 기본 예시는 무작위 초기화 모델을 사용하므로 특징맵의 의미보다 텐서 모양과 구조 변화에 집중합니다. 학습된 체크포인트를 불러오면 실제 분류에 유용한 패턴으로 변화한 특징맵을 관찰할 수 있습니다.

## 학습 목표와 관찰 원칙

이 노트북을 마치면 다음을 설명할 수 있어야 합니다.

1. 작은 합성곱 필터를 이미지 전체에 반복 적용하는 이유를 설명합니다.
2. 채널, 높이, 너비로 구성된 특징맵의 shape을 읽습니다.
3. Max pooling과 average pooling이 서로 다른 정보를 남기는 방식을 비교합니다.
4. 깊은 층일수록 더 넓은 입력 문맥을 보는 이유를 수용영역으로 설명합니다.
5. 파라미터 수와 모델의 구조적 효율성을 구분합니다.

각 그림에서는 색이 밝다는 사실만으로 좋은 특징이라고 판단하지 마세요. 채널마다 값의 범위가 다를 수 있으므로 **어느 위치에 반응하는지, 공간 크기가 어떻게 변하는지, 층이 깊어질수록 패턴이 어떻게 조합되는지**를 봅니다.

In [ ]:
import torch
import matplotlib.pyplot as plt
from torchinfo import summary

from cifar10_lab import (
    CIFAR10_MEAN, CIFAR10_STD, create_model, detect_environment,
    load_cifar10_data, resolve_data_dir, set_global_seed,
)
from cifar10_lab.cnn_visualization import (
    collect_feature_maps, plot_feature_map_progression,
    plot_pooling_comparison, plot_receptive_field_progression,
    receptive_field_stages,
)

set_global_seed(42, deterministic=True)
runtime = detect_environment()
print('Runtime:', runtime)

## 1. 한 장의 이미지에서 시작하기

CIFAR-10 이미지는 `(채널, 높이, 너비) = (3, 32, 32)` 형태입니다. 세 채널은 RGB 색 정보를 담습니다.

- MLP는 이를 3,072개의 긴 벡터로 펼쳐 위치 관계를 모델 구조에 명시적으로 남기지 않습니다.
- CNN은 높이와 너비 축을 유지한 채 작은 필터를 이동시켜 가까운 픽셀의 관계를 먼저 학습합니다.
- 입력은 채널별 평균과 표준편차로 정규화됩니다. 시각화할 때는 사람이 볼 수 있도록 정규화를 되돌립니다.

아래 셀은 Test 데이터에서 이미지 한 장을 가져옵니다. 이후 모든 실습이 같은 이미지를 사용하므로 층별 변화를 연속적으로 비교할 수 있습니다.

In [ ]:
# 데이터가 없다면 torchvision이 처음 한 번 자동 다운로드합니다.
_, _, testloader, classes = load_cifar10_data(
    batch_size=1, seed=42, num_workers=runtime.num_workers,
    pin_memory=runtime.pin_memory, data_root=resolve_data_dir(),
    max_train_samples=1, max_val_samples=1, max_test_samples=1,
)
image, label = next(iter(testloader))
image = image.to(runtime.device)

# 화면에 표시할 때만 정규화를 되돌립니다. 모델에는 정규화된 입력을 넣습니다.
mean = torch.tensor(CIFAR10_MEAN).view(3, 1, 1)
std = torch.tensor(CIFAR10_STD).view(3, 1, 1)
display_image = (image[0].cpu() * std + mean).clamp(0, 1)
plt.figure(figsize=(3.2, 3.2))
plt.imshow(display_image.permute(1, 2, 0))
plt.title(f'Input image: {classes[int(label[0])]}')
plt.axis('off')
plt.show()

### 이미지 출력 확인

제목에는 정답 클래스가 표시됩니다. 이미지가 약간 흐리거나 색이 원본과 다르게 보인다면 정규화 복원과 작은 `32×32` 해상도의 영향일 수 있습니다. 이 이미지는 모델이 실제로 받는 텐서와 동일한 표본입니다.

## 2. CIFAR-10용 AlexNet 구조

원래 AlexNet은 큰 ImageNet 이미지를 대상으로 설계되었습니다. 이 프로젝트의 구현은 `32×32` 입력에 맞춰 첫 합성곱과 pooling 크기를 조정하지만, 다음 핵심 흐름은 유지합니다.

1. **Conv2d**가 작은 지역 패턴을 찾습니다.
2. **ReLU**가 비선형성을 추가합니다.
3. **MaxPool2d**가 공간 크기를 줄이고 강한 반응을 요약합니다.
4. 여러 합성곱 층이 단순 패턴을 더 복합적인 패턴으로 조합합니다.
5. 마지막 분류기가 특징을 10개 클래스 점수로 변환합니다.

`torchinfo.summary`에서는 각 층의 출력 shape과 파라미터 수를 확인합니다. 배치 차원을 제외하고 채널 수가 늘고 공간 크기가 줄어드는 흐름을 따라가세요.

In [ ]:
# 출력 shape은 배치, 채널, 높이, 너비 순서로 읽습니다.
alexnet = create_model('alexnet', num_classes=10, image_size=32).to(runtime.device).eval()
model_summary = summary(alexnet, input_size=(1, 3, 32, 32), depth=3, verbose=0)
print(model_summary)

### 모델 요약표 읽는 순서

1. 각 층의 입력·출력 shape을 확인합니다.
2. Conv2d 뒤에서 채널 수가 어떻게 변하는지 봅니다.
3. Pooling 뒤에서 높이와 너비가 줄어드는지 봅니다.
4. 마지막 출력이 `(1, 10)`인지 확인합니다.
5. 파라미터 대부분이 합성곱과 분류기 중 어디에 있는지 비교합니다.

shape을 읽는 습관은 모델 연결 오류를 찾는 가장 기본적인 디버깅 방법입니다.

## 3. 특징맵: 한 이미지를 여러 관점으로 보기

합성곱 필터 하나는 하나의 출력 채널을 만듭니다. 여러 필터를 사용하면 같은 입력을 서로 다른 관점으로 본 여러 특징맵이 생성됩니다.

합성곱 출력 크기는 padding과 dilation을 단순화하면 다음처럼 계산할 수 있습니다.

$$H_{out}=\left\lfloor\frac{H_{in}+2P-K}{S}\right\rfloor+1$$

$K$는 kernel 크기, $S$는 stride, $P$는 padding입니다. `hook`은 순전파 계산을 바꾸지 않고 중간 출력만 기록합니다.

- 얕은 층은 경계, 방향, 색 변화 같은 국소 패턴에 반응할 수 있습니다.
- 깊은 층은 앞 층의 특징을 조합해 더 복합적인 패턴을 표현할 수 있습니다.
- Pooling 뒤에는 높이와 너비가 줄어들어 위치 정밀도보다 요약된 의미가 강조됩니다.

In [ ]:
# Conv2d와 MaxPool2d를 통과할 때의 중간 출력을 hook으로 기록합니다.
feature_records = collect_feature_maps(alexnet, image, max_layers=6)
for record in feature_records:
    print(record['name'], record['type'], '->', record['shape'])
plot_feature_map_progression(feature_records, max_channels=4)
plt.show()

### 특징맵 그림 읽기

각 행은 하나의 층, 각 열은 그 층의 서로 다른 채널입니다. 행 이름 옆 shape의 첫 숫자는 채널 수이고 뒤의 두 숫자는 공간 크기입니다. 같은 행의 채널들이 서로 다른 위치에 반응한다면 필터가 서로 다른 패턴을 보고 있다는 뜻입니다.

무작위 초기화 모델에서는 의미 있는 물체 부위보다 경계와 잡음 같은 반응이 보일 수 있습니다. 학습 후에는 클래스 구분에 유용한 패턴이 상대적으로 조직화됩니다.

## 4. Pooling: 작게 만들면서 무엇을 남길 것인가

Pooling은 작은 창 안의 여러 값을 하나로 줄입니다. 예를 들어 `2×2` 영역 `[1, 2; 3, 8]`에서 Max pooling은 `8`, average pooling은 `3.5`를 남깁니다.

| 방식 | 남기는 값 | 강조되는 정보 | 손실되는 정보 |
|---|---|---|---|
| Max pooling | 영역의 최댓값 | 가장 강한 특징의 존재 | 주변 값의 평균적 분포 |
| Average pooling | 영역의 평균 | 전체적인 밝기와 경향 | 매우 강한 국소 반응 |

두 방식 모두 공간 해상도와 다음 층의 계산량을 줄입니다. 그림에서 원본과 두 결과의 윤곽, 밝기, 세부 정보가 어떻게 달라지는지 비교하세요.

In [ ]:
# 같은 입력에 두 pooling을 적용해 남는 정보의 차이만 비교합니다.
plot_pooling_comparison(image)
plt.show()

### Pooling 비교 체크리스트

- Max pooling 결과에서 강한 경계가 더 두드러지는지 확인합니다.
- Average pooling 결과가 더 부드럽게 보이는지 확인합니다.
- 두 결과 모두 원본보다 공간 해상도가 작아졌는지 확인합니다.
- 작은 물체에서는 지나친 축소가 중요한 정보를 없앨 수 있다는 점을 생각해 봅니다.

## 5. 수용영역: 깊어질수록 더 넓게 보기

수용영역은 특정 특징값 하나에 영향을 줄 수 있는 원본 이미지 영역입니다. 첫 `3×3` 합성곱의 출력 하나는 입력의 `3×3` 영역을 봅니다. 다음 `3×3` 합성곱은 앞 층의 여러 값을 조합하므로 원본 기준으로 더 넓은 영역을 보게 됩니다.

층 $l$의 수용영역 $r_l$과 입력 위 이동 간격 $j_l$은 다음처럼 누적됩니다.

$$r_l = r_{l-1} + (k_l-1)j_{l-1}$$
$$j_l = j_{l-1}s_l$$

kernel이 커지거나 stride가 커지면 수용영역이 더 빠르게 넓어집니다. 그래프에서 합성곱과 pooling을 지날 때 수용영역이 얼마나 증가하는지 확인하세요.

In [ ]:
# 각 층의 kernel과 stride가 누적 수용영역을 어떻게 키우는지 추적합니다.
stages = receptive_field_stages(alexnet)
for stage in stages:
    print(stage)
plot_receptive_field_progression(alexnet)
plt.show()

### 수용영역 출력 해석

출력된 각 사전에는 층 이름, kernel, stride, 현재 수용영역이 기록됩니다. 그래프가 계단식으로 증가하는 이유는 층마다 kernel과 stride가 다르기 때문입니다. 이론적 수용영역이 이미지 전체보다 커질 수도 있으며, 이는 padding을 포함한 계산 경로의 범위를 뜻합니다.

## 6. 퍼셉트론, MLP, AlexNet, ResNet의 크기 비교

파라미터 수는 모델 용량과 저장 크기를 짐작하게 하지만, 그 자체가 성능 순위는 아닙니다.

- 퍼셉트론은 작고 빠르지만 하나의 선형 경계만 만듭니다.
- MLP는 비선형 표현을 얻지만 모든 픽셀을 펼쳐 연결하므로 위치 공유가 없습니다.
- AlexNet은 합성곱 필터를 모든 위치에 공유해 이미지에 적합한 계층 구조를 만듭니다.
- ResNet은 잔차 연결을 이용해 더 깊은 모델의 최적화를 안정화합니다.

막대그래프는 파라미터 수의 차이를 보여 줍니다. 정확도와 추론 시간까지 함께 측정해야 실제 목적에 맞는 모델을 선택할 수 있습니다.

In [ ]:
# 정확도가 아니라 구조적 크기 차이를 보기 위한 파라미터 비교입니다.
model_ids = ('perceptron', 'mlp', 'alexnet', 'resnet18')
parameter_counts = []
for model_id in model_ids:
    model = create_model(model_id, num_classes=10, image_size=32)
    parameter_counts.append(sum(parameter.numel() for parameter in model.parameters()))
figure, axis = plt.subplots(figsize=(8, 4))
bars = axis.bar(model_ids, [value / 1_000_000 for value in parameter_counts])
axis.bar_label(bars, labels=[f'{value / 1_000_000:.2f}M' for value in parameter_counts])
axis.set_ylabel('parameters (millions)')
axis.set_title('Architecture changes matter more than parameter count alone')
figure.tight_layout()
plt.show()

### 파라미터 그래프 해석

파라미터가 많은 모델은 더 많은 패턴을 저장할 가능성이 있지만 더 큰 체크포인트와 메모리를 요구합니다. 반대로 파라미터가 적다고 항상 추론이 빠른 것도 아닙니다. 실제 비교에서는 연산 종류, 하드웨어, 배치 크기도 함께 고려해야 합니다.

## 정리와 다음 실험

이 노트북에서 확인한 핵심은 다음과 같습니다.

1. 합성곱은 지역 연결과 가중치 공유를 이용합니다.
2. 한 층의 여러 필터가 여러 특징맵 채널을 만듭니다.
3. Pooling은 계산량을 줄이는 동시에 어떤 정보를 남길지 결정합니다.
4. 깊은 층은 넓은 수용영역을 통해 더 큰 문맥을 조합합니다.
5. 모델 크기와 구조적 효율성은 구분해서 비교해야 합니다.

### 이어서 해볼 실험

```powershell
# AlexNet 빠른 학습
cifar10-lab train --model alexnet --quick --save-plots

# 학습된 체크포인트의 특징맵 저장
cifar10-lab visualize-cnn --model alexnet --checkpoint-dir 체크포인트폴더
```

무작위 초기화 특징맵과 학습된 특징맵을 같은 이미지로 비교해 보세요. 그다음 `main.ipynb`에서 AlexNet과 현대 CNN·Transformer를 같은 조건으로 비교합니다.